# Notebook 07: Retention Strength Scoring & Final Segmentation Analysis

## Project: Customer Engagement & Product Utilization Analytics for Retention Strategy

This notebook focuses on building the final retention strength scoring layer and customer segmentation logic for the banking retention analytics project.

The main purpose of this notebook is to evaluate how strongly each customer is connected with the bank based on engagement behavior, product usage, financial value, relationship strength, and retention priority signals generated in the previous notebooks.

---

## Notebook Objective

The objective of Notebook 07 is to create a dashboard-ready analytical layer that helps identify:

- Customers with strong retention signals
- Customers with weak or unstable relationship strength
- Sticky customers with strong loyalty behavior
- High-risk customers requiring retention attention
- Final customer segments for retention strategy and dashboard use

---

## Business Context

Customer churn is not driven by one factor alone. A customer may have a high balance but still churn if engagement is weak. Similarly, a customer using multiple products may show stronger retention if engagement and relationship strength are also high.

This notebook combines multiple retention signals into a structured scoring and segmentation framework. The goal is to support better decision-making for customer retention, risk monitoring, and dashboard-level prioritization.

---

## Key Questions Answered in This Notebook

1. How does Relationship Strength Index vary across customers?
2. Which relationship strength tiers have the highest churn risk?
3. Do active customers with strong relationship strength retain better?
4. Which customers can be classified as sticky customers?
5. Which customers require priority retention action?
6. How can retention strength be converted into dashboard-ready KPIs and segments?

---

## Inputs Used

This notebook uses the latest enriched dataset generated from Notebook 06:

`retention_priority_master_dataset.csv`

This dataset already includes:

- Engagement status
- Product depth segmentation
- Balance and salary segmentation
- High-value customer flags
- High-value disengagement flags
- At-risk premium customer flags
- Salary-balance mismatch flags
- Customer risk segments
- Relationship Strength Index
- Relationship Strength Tier
- Retention Priority Segment

---

## Expected Output

Notebook 07 will generate retention strength scoring outputs for dashboard development, including:

- Relationship strength summary
- Churn by relationship strength tier
- Engagement and relationship strength retention analysis
- Sticky customer profile
- Retention strength KPI summary
- Final retention strength segments
- Dashboard-ready CSV files

---

## Dashboard Module Supported

The outputs from this notebook will support the Streamlit dashboard module:

`Retention Strength Scoring & Final Customer Segmentation Overview`

---

## Notebook Flow

1. Import required libraries  
2. Define project paths  
3. Load latest working dataset  
4. Validate dataset shape and required columns  
5. Review Relationship Strength Index distribution  
6. Analyze churn by relationship strength tier  
7. Analyze retention by engagement and relationship strength tier  
8. Identify sticky customer profiles  
9. Create retention strength KPI summary  
10. Prepare final segmentation layer for dashboard use

### Import Required Libraries

In [28]:
# Step 7A: Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


### Define Project Paths

In [29]:
# Step 7B: Define project paths

BASE_DIR = Path.cwd().parent

# Latest enriched dataset from Notebook 06
RETENTION_PRIORITY_DATA_PATH = (
    BASE_DIR / "data" / "processed" / "retention_priority_analysis" / "retention_priority_master_dataset.csv"
)

# Output folder for Notebook 07
RETENTION_STRENGTH_OUTPUT_DIR = (
    BASE_DIR / "data" / "processed" / "retention_strength_scoring"
)

RETENTION_STRENGTH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base Directory:", BASE_DIR)
print("Input Dataset Path:", RETENTION_PRIORITY_DATA_PATH)
print("Output Directory:", RETENTION_STRENGTH_OUTPUT_DIR)

Base Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy
Input Dataset Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\retention_priority_analysis\retention_priority_master_dataset.csv
Output Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\retention_strength_scoring


### Load Latest Working Dataset

In [30]:
# Step 7C: Load latest working dataset

df = pd.read_csv(RETENTION_PRIORITY_DATA_PATH)

# Preview dataset
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Churn_Status,Engagement_Status,Credit_Card_Status,Product_Depth_Segment,Balance_Segment,Salary_Segment,Age_Segment,Tenure_Segment,High_Value_Customer_Flag,High_Value_Customer_Status,High_Value_Disengaged_Flag,High_Value_Disengaged_Status,At_Risk_Premium_Flag,At_Risk_Premium_Status,Salary_Balance_Mismatch_Flag,Salary_Balance_Mismatch_Status,Customer_Risk_Segment,Relationship_Strength_Index,Relationship_Strength_Tier,High_Balance_Flag,High_Balance_Status,Retention_Priority_Segment
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,"101,348.88",1,Churned,Active Member,Has Credit Card,Single Product,Zero Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship,0,Not High Balance,Already Churned
1,2025,15647311,Hill,608,Spain,Female,41,1,"83,807.86",1,0,1,"112,542.58",0,Retained,Active Member,No Credit Card,Single Product,Medium Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship,0,Not High Balance,Low Priority
2,2025,15619304,Onio,502,France,Female,42,8,"159,660.80",3,1,0,"113,931.57",1,Churned,Inactive Member,Has Credit Card,High Product Load,Premium Balance,High Salary,Mid-Age Customer,Established Relationship,1,High Value Customer,1,High-Value Disengaged,1,At-Risk Premium Customer,0,No Salary-Balance Mismatch,At-Risk Premium,50,Weak Relationship,1,High Balance Customer,Already Churned
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,"93,826.63",0,Retained,Inactive Member,No Credit Card,Optimal Product Depth,Zero Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,35,Weak Relationship,0,Not High Balance,Medium Priority
4,2025,15737888,Mitchell,850,Spain,Female,43,2,"125,510.82",1,1,1,"79,084.10",0,Retained,Active Member,Has Credit Card,Single Product,High Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,75,Moderate Relationship,0,Not High Balance,Low Priority


### Dataset Shape + Column Validation

In [31]:
# Step 7D: Dataset shape and column validation

print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]:,}")

print("\nAvailable Columns:")
for col in df.columns:
    print("-", col)

Total Rows: 10,000
Total Columns: 36

Available Columns:
- Year
- CustomerId
- Surname
- CreditScore
- Geography
- Gender
- Age
- Tenure
- Balance
- NumOfProducts
- HasCrCard
- IsActiveMember
- EstimatedSalary
- Exited
- Churn_Status
- Engagement_Status
- Credit_Card_Status
- Product_Depth_Segment
- Balance_Segment
- Salary_Segment
- Age_Segment
- Tenure_Segment
- High_Value_Customer_Flag
- High_Value_Customer_Status
- High_Value_Disengaged_Flag
- High_Value_Disengaged_Status
- At_Risk_Premium_Flag
- At_Risk_Premium_Status
- Salary_Balance_Mismatch_Flag
- Salary_Balance_Mismatch_Status
- Customer_Risk_Segment
- Relationship_Strength_Index
- Relationship_Strength_Tier
- High_Balance_Flag
- High_Balance_Status
- Retention_Priority_Segment


### Validate Required Columns

In [32]:
# Step 7E: Validate required columns for retention strength scoring

required_columns = [
    "CustomerId",
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited",
    "Engagement_Status",
    "Credit_Card_Status",
    "Product_Depth_Segment",
    "Balance_Segment",
    "Salary_Segment",
    "High_Value_Customer_Flag",
    "High_Value_Customer_Status",
    "High_Value_Disengaged_Flag",
    "High_Value_Disengaged_Status",
    "At_Risk_Premium_Flag",
    "At_Risk_Premium_Status",
    "Salary_Balance_Mismatch_Flag",
    "Salary_Balance_Mismatch_Status",
    "Customer_Risk_Segment",
    "Relationship_Strength_Index",
    "Relationship_Strength_Tier",
    "Retention_Priority_Segment"
]

missing_required_columns = [col for col in required_columns if col not in df.columns]

if len(missing_required_columns) == 0:
    print("All required columns are available.")
else:
    print("Missing required columns:")
    print(missing_required_columns)

All required columns are available.


### Review Relationship Strength Index Distribution

In [33]:
# Step 7F: Review Relationship Strength Index distribution

rsi_summary = pd.DataFrame({
    "Metric": [
        "Minimum RSI",
        "Maximum RSI",
        "Average RSI",
        "Median RSI",
        "Standard Deviation RSI"
    ],
    "Value": [
        df["Relationship_Strength_Index"].min(),
        df["Relationship_Strength_Index"].max(),
        df["Relationship_Strength_Index"].mean(),
        df["Relationship_Strength_Index"].median(),
        df["Relationship_Strength_Index"].std()
    ]
})

rsi_summary["Value"] = rsi_summary["Value"].round(2)

rsi_summary

,Metric,Value
0,Minimum RSI,10.00
1,Maximum RSI,110.00
2,Average RSI,64.44
3,Median RSI,65.00
4,Standard Deviation RSI,19.81


### Analyze Churn by Relationship Strength Tier

In [34]:
# Step 7G: Analyze churn by relationship strength tier

rsi_tier_churn = (
    df
    .groupby("Relationship_Strength_Tier")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

rsi_tier_churn["Churn_Rate"] = (
    rsi_tier_churn["Churned_Customers"] / rsi_tier_churn["Customer_Count"] * 100
).round(2)

rsi_tier_churn["Retention_Rate"] = (
    rsi_tier_churn["Retained_Customers"] / rsi_tier_churn["Customer_Count"] * 100
).round(2)

for col in ["Avg_Relationship_Strength", "Avg_Balance", "Avg_Num_Products"]:
    rsi_tier_churn[col] = rsi_tier_churn[col].round(2)

rsi_tier_churn = rsi_tier_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

rsi_tier_churn

,Relationship_Strength_Tier,Customer_Count,Churned_Customers,Retained_Customers,Avg_Relationship_Strength,Avg_Balance,Avg_Num_Products,Churn_Rate,Retention_Rate
2,Very Weak Relationship,262,132,130,22.12,"9,781.80",1.39,50.38,49.62
3,Weak Relationship,2869,854,2015,43.17,"67,479.35",1.41,29.77,70.23
0,Moderate Relationship,3996,733,3263,65.48,"78,214.53",1.51,18.34,81.66
1,Strong Relationship,2873,318,2555,88.11,"89,158.56",1.69,11.07,88.93


### Analyze Retention by Engagement + RSI Tier

In [35]:
# Step 7H: Analyze retention by engagement status and relationship strength tier

engagement_rsi_retention = (
    df
    .groupby(["Engagement_Status", "Relationship_Strength_Tier"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

engagement_rsi_retention["Churn_Rate"] = (
    engagement_rsi_retention["Churned_Customers"] / engagement_rsi_retention["Customer_Count"] * 100
).round(2)

engagement_rsi_retention["Retention_Rate"] = (
    engagement_rsi_retention["Retained_Customers"] / engagement_rsi_retention["Customer_Count"] * 100
).round(2)

for col in ["Avg_Relationship_Strength", "Avg_Balance", "Avg_Num_Products"]:
    engagement_rsi_retention[col] = engagement_rsi_retention[col].round(2)

engagement_rsi_retention = engagement_rsi_retention.sort_values(
    by=["Engagement_Status", "Churn_Rate"],
    ascending=[True, False]
)

engagement_rsi_retention

,Engagement_Status,Relationship_Strength_Tier,Customer_Count,Churned_Customers,Retained_Customers,Avg_Relationship_Strength,Avg_Balance,Avg_Num_Products,Churn_Rate,Retention_Rate
2,Active Member,Weak Relationship,91,42,49,47.42,0.00,1.51,46.15,53.85
0,Active Member,Moderate Relationship,2267,391,1876,68.91,"64,070.98",1.36,17.25,82.75
1,Active Member,Strong Relationship,2793,302,2491,88.34,"87,928.89",1.68,10.81,89.19
5,Inactive Member,Very Weak Relationship,262,132,130,22.12,"9,781.80",1.39,50.38,49.62
6,Inactive Member,Weak Relationship,2778,812,1966,43.03,"69,689.79",1.41,29.23,70.77
4,Inactive Member,Strong Relationship,80,16,64,80.00,"132,089.34",2.00,20.00,80.00
3,Inactive Member,Moderate Relationship,1729,342,1387,60.97,"96,759.03",1.71,19.78,80.22


### Identify Sticky Customer Profile

In [36]:
# Step 7I: Identify sticky customer profile

sticky_customer_filter = (
    (df["Exited"] == 0) &
    (df["Engagement_Status"] == "Active Member") &
    (df["Relationship_Strength_Tier"].isin(["Strong Relationship", "Moderate Relationship"])) &
    (df["Product_Depth_Segment"].isin(["Optimal Product Depth", "High Product Load"]))
)

sticky_customers = df[sticky_customer_filter].copy()

sticky_customer_summary = {
    "Sticky Customers": len(sticky_customers),
    "Sticky Customer Share": round(len(sticky_customers) / len(df) * 100, 2),
    "Average Relationship Strength": round(sticky_customers["Relationship_Strength_Index"].mean(), 2),
    "Average Balance": round(sticky_customers["Balance"].mean(), 2),
    "Average Salary": round(sticky_customers["EstimatedSalary"].mean(), 2),
    "Average Number of Products": round(sticky_customers["NumOfProducts"].mean(), 2),
    "Credit Card Ownership Rate": round(sticky_customers["HasCrCard"].mean() * 100, 2)
}

sticky_customer_summary

{'Sticky Customers': 2330,
 'Sticky Customer Share': 23.3,
 'Average Relationship Strength': np.float64(85.79),
 'Average Balance': np.float64(49302.34),
 'Average Salary': np.float64(100665.27),
 'Average Number of Products': np.float64(2.01),
 'Credit Card Ownership Rate': np.float64(71.33)}

### Create First Retention Strength KPI Summary

In [37]:
# Step 7J: Create first retention strength KPI summary

strong_relationship_retention = rsi_tier_churn.loc[
    rsi_tier_churn["Relationship_Strength_Tier"] == "Strong Relationship",
    "Retention_Rate"
].iloc[0]

weak_relationship_churn = rsi_tier_churn.loc[
    rsi_tier_churn["Relationship_Strength_Tier"] == "Weak Relationship",
    "Churn_Rate"
].iloc[0]

very_weak_relationship_churn = rsi_tier_churn.loc[
    rsi_tier_churn["Relationship_Strength_Tier"] == "Very Weak Relationship",
    "Churn_Rate"
].iloc[0]

active_strong_retention = engagement_rsi_retention.loc[
    (engagement_rsi_retention["Engagement_Status"] == "Active Member") &
    (engagement_rsi_retention["Relationship_Strength_Tier"] == "Strong Relationship"),
    "Retention_Rate"
].iloc[0]

inactive_weak_churn = engagement_rsi_retention.loc[
    (engagement_rsi_retention["Engagement_Status"] == "Inactive Member") &
    (engagement_rsi_retention["Relationship_Strength_Tier"] == "Weak Relationship"),
    "Churn_Rate"
].iloc[0]

retention_strength_kpi_summary = pd.DataFrame({
    "KPI": [
        "Strong Relationship Retention Rate",
        "Weak Relationship Churn Rate",
        "Very Weak Relationship Churn Rate",
        "Active Strong Relationship Retention Rate",
        "Inactive Weak Relationship Churn Rate",
        "Sticky Customer Count",
        "Sticky Customer Share"
    ],
    "Value": [
        f"{strong_relationship_retention:.2f}%",
        f"{weak_relationship_churn:.2f}%",
        f"{very_weak_relationship_churn:.2f}%",
        f"{active_strong_retention:.2f}%",
        f"{inactive_weak_churn:.2f}%",
        f"{sticky_customer_summary['Sticky Customers']:,}",
        f"{sticky_customer_summary['Sticky Customer Share']:.2f}%"
    ],
    "Business Meaning": [
        "Retention rate among customers with strong relationship strength",
        "Churn risk among customers with weak relationship strength",
        "Churn risk among customers with very weak relationship strength",
        "Best retention profile using engagement and relationship strength together",
        "High-risk profile combining inactivity and weak relationship strength",
        "Number of retained customers with strong behavioral loyalty signals",
        "Share of total customers classified as sticky customers"
    ]
})

retention_strength_kpi_summary

,KPI,Value,Business Meaning
0,Strong Relationship Retention Rate,88.93%,Retention rate among customers with strong rel...
1,Weak Relationship Churn Rate,29.77%,Churn risk among customers with weak relations...
2,Very Weak Relationship Churn Rate,50.38%,Churn risk among customers with very weak rela...
3,Active Strong Relationship Retention Rate,89.19%,Best retention profile using engagement and re...
4,Inactive Weak Relationship Churn Rate,29.23%,High-risk profile combining inactivity and wea...
5,Sticky Customer Count,"2,330",Number of retained customers with strong behav...
6,Sticky Customer Share,23.30%,Share of total customers classified as sticky ...


## Retention Strength Score Creation

This step creates a final retention strength score using engagement, product depth, relationship strength, credit card ownership, financial value, and churn status.

The goal is to create a dashboard-ready scoring metric that can help rank customers by retention quality and risk.

In [38]:
# Step 7K: Create final Retention Strength Score

df["Retention_Strength_Score"] = (
    (df["IsActiveMember"] * 20) +
    (df["HasCrCard"] * 10) +
    (df["NumOfProducts"].clip(upper=3) * 10) +
    (df["Relationship_Strength_Index"] * 0.40) +
    (df["High_Value_Customer_Flag"] * 10) -
    (df["Exited"] * 25)
)

# Safe Patch: Keep score within clean dashboard-friendly 0–100 range
df["Retention_Strength_Score"] = (
    df["Retention_Strength_Score"]
    .clip(lower=0, upper=100)
    .round(2)
)

retention_score_summary = pd.DataFrame({
    "Metric": [
        "Minimum Retention Strength Score",
        "Maximum Retention Strength Score",
        "Average Retention Strength Score",
        "Median Retention Strength Score",
        "Score Standard Deviation"
    ],
    "Value": [
        df["Retention_Strength_Score"].min(),
        df["Retention_Strength_Score"].max(),
        df["Retention_Strength_Score"].mean(),
        df["Retention_Strength_Score"].median(),
        df["Retention_Strength_Score"].std()
    ]
})

retention_score_summary["Value"] = retention_score_summary["Value"].round(2)

retention_score_summary

,Metric,Value
0,Minimum Retention Strength Score,0.00
1,Maximum Retention Strength Score,100.00
2,Average Retention Strength Score,57.63
3,Median Retention Strength Score,58.00
4,Score Standard Deviation,23.75


## Retention Strength Tier Creation

This step converts the numerical Retention Strength Score into business-friendly retention strength tiers.

These tiers will help the dashboard separate strong, moderate, weak, and critical retention profiles.

In [39]:
# Step 7L: Create retention strength tiers

def assign_retention_strength_tier(score):
    if score >= 90:
        return "Very Strong Retention"
    elif score >= 75:
        return "Strong Retention"
    elif score >= 55:
        return "Moderate Retention"
    elif score >= 35:
        return "Weak Retention"
    else:
        return "Critical Retention Risk"

df["Retention_Strength_Tier"] = df["Retention_Strength_Score"].apply(assign_retention_strength_tier)

retention_strength_tier_distribution = (
    df["Retention_Strength_Tier"]
    .value_counts()
    .reset_index()
)

retention_strength_tier_distribution.columns = [
    "Retention Strength Tier",
    "Customer Count"
]

retention_strength_tier_distribution["Customer %"] = (
    retention_strength_tier_distribution["Customer Count"] / len(df) * 100
).round(2)

retention_strength_tier_distribution

,Retention Strength Tier,Customer Count,Customer %
0,Weak Retention,2857,28.57
1,Moderate Retention,2640,26.40
2,Strong Retention,1953,19.53
3,Critical Retention Risk,1776,17.76
4,Very Strong Retention,774,7.74


## Retention Strength Tier Churn Analysis

This step measures churn and retention behavior across retention strength tiers.

The objective is to verify whether lower retention strength tiers show higher churn risk.

In [40]:
# Step 7M: Analyze churn by retention strength tier

retention_strength_tier_churn = (
    df
    .groupby("Retention_Strength_Tier")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Retention_Strength_Score=("Retention_Strength_Score", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Balance=("Balance", "mean")
    )
    .reset_index()
)

retention_strength_tier_churn["Churn_Rate"] = (
    retention_strength_tier_churn["Churned_Customers"] / retention_strength_tier_churn["Customer_Count"] * 100
).round(2)

retention_strength_tier_churn["Retention_Rate"] = (
    retention_strength_tier_churn["Retained_Customers"] / retention_strength_tier_churn["Customer_Count"] * 100
).round(2)

for col in [
    "Avg_Retention_Strength_Score",
    "Avg_Relationship_Strength",
    "Avg_Num_Products",
    "Avg_Balance"
]:
    retention_strength_tier_churn[col] = retention_strength_tier_churn[col].round(2)

retention_strength_tier_churn = retention_strength_tier_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

retention_strength_tier_churn

,Retention_Strength_Tier,Customer_Count,Churned_Customers,Retained_Customers,Avg_Retention_Strength_Score,Avg_Relationship_Strength,Avg_Num_Products,Avg_Balance,Churn_Rate,Retention_Rate
0,Critical Retention Risk,1776,1225,551,20.92,40.69,1.23,"76,478.51",68.98,31.02
4,Weak Retention,2857,482,2375,45.95,52.81,1.51,"72,244.52",16.87,83.13
1,Moderate Retention,2640,286,2354,65.61,70.54,1.52,"82,000.52",10.83,89.17
2,Strong Retention,1953,44,1909,82.30,82.62,1.66,"69,709.13",2.25,97.75
3,Very Strong Retention,774,0,774,95.49,95.23,2.01,"90,448.53",0.00,100.00


## Engagement and Retention Strength Matrix

This step compares engagement status with retention strength tiers.

This matrix helps identify active customers with weak retention signals and inactive customers with strong potential value.

In [41]:
# Step 7N: Engagement and retention strength matrix

engagement_retention_strength_matrix = (
    df
    .groupby(["Engagement_Status", "Retention_Strength_Tier"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Retention_Strength_Score=("Retention_Strength_Score", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

engagement_retention_strength_matrix["Churn_Rate"] = (
    engagement_retention_strength_matrix["Churned_Customers"] /
    engagement_retention_strength_matrix["Customer_Count"] * 100
).round(2)

engagement_retention_strength_matrix["Retention_Rate"] = (
    engagement_retention_strength_matrix["Retained_Customers"] /
    engagement_retention_strength_matrix["Customer_Count"] * 100
).round(2)

for col in [
    "Avg_Retention_Strength_Score",
    "Avg_Relationship_Strength",
    "Avg_Num_Products"
]:
    engagement_retention_strength_matrix[col] = engagement_retention_strength_matrix[col].round(2)

engagement_retention_strength_matrix = engagement_retention_strength_matrix.sort_values(
    by=["Engagement_Status", "Churn_Rate"],
    ascending=[True, False]
)

engagement_retention_strength_matrix

,Engagement_Status,Retention_Strength_Tier,Customer_Count,Churned_Customers,Retained_Customers,Avg_Retention_Strength_Score,Avg_Relationship_Strength,Avg_Num_Products,Churn_Rate,Retention_Rate
0,Active Member,Critical Retention Risk,75,75,0,28.49,57.40,1.00,100.00,0.00
4,Active Member,Weak Retention,458,330,128,46.30,66.07,1.15,72.05,27.95
1,Active Member,Moderate Retention,1891,286,1605,66.65,72.78,1.33,15.12,84.88
2,Active Member,Strong Retention,1953,44,1909,82.30,82.62,1.66,2.25,97.75
3,Active Member,Very Strong Retention,774,0,774,95.49,95.23,2.01,0.00,100.00
5,Inactive Member,Critical Retention Risk,1701,1150,551,20.59,39.96,1.24,67.61,32.39
7,Inactive Member,Weak Retention,2399,152,2247,45.88,50.28,1.57,6.34,93.66
6,Inactive Member,Moderate Retention,749,0,749,62.98,64.87,2.01,0.00,100.00


## Product Depth and Retention Strength Matrix

This step compares product depth with retention strength tiers.

The goal is to check whether customers with optimal product usage also show stronger retention strength.

In [42]:
# Step 7O: Product depth and retention strength matrix

product_depth_retention_strength_matrix = (
    df
    .groupby(["Product_Depth_Segment", "Retention_Strength_Tier"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Retention_Strength_Score=("Retention_Strength_Score", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Balance=("Balance", "mean")
    )
    .reset_index()
)

product_depth_retention_strength_matrix["Churn_Rate"] = (
    product_depth_retention_strength_matrix["Churned_Customers"] /
    product_depth_retention_strength_matrix["Customer_Count"] * 100
).round(2)

product_depth_retention_strength_matrix["Retention_Rate"] = (
    product_depth_retention_strength_matrix["Retained_Customers"] /
    product_depth_retention_strength_matrix["Customer_Count"] * 100
).round(2)

for col in [
    "Avg_Retention_Strength_Score",
    "Avg_Relationship_Strength",
    "Avg_Num_Products",
    "Avg_Balance"
]:
    product_depth_retention_strength_matrix[col] = product_depth_retention_strength_matrix[col].round(2)

product_depth_retention_strength_matrix = product_depth_retention_strength_matrix.sort_values(
    by=["Product_Depth_Segment", "Churn_Rate"],
    ascending=[True, False]
)

product_depth_retention_strength_matrix

,Product_Depth_Segment,Retention_Strength_Tier,Customer_Count,Churned_Customers,Retained_Customers,Avg_Retention_Strength_Score,Avg_Relationship_Strength,Avg_Num_Products,Avg_Balance,Churn_Rate,Retention_Rate
0,High Product Load,Critical Retention Risk,99,99,0,25.06,30.71,3.19,"68,338.83",100.00,0.00
4,High Product Load,Weak Retention,101,88,13,43.78,45.35,3.16,"90,487.27",87.13,12.87
1,High Product Load,Moderate Retention,86,73,13,64.38,63.14,3.21,"73,731.85",84.88,15.12
2,High Product Load,Strong Retention,35,20,15,79.54,70.29,3.20,"81,769.95",57.14,42.86
3,High Product Load,Very Strong Retention,5,0,5,98.00,70.00,3.00,"117,651.63",0.00,100.00
5,Optimal Product Depth,Critical Retention Risk,197,127,70,28.13,48.81,2.00,"47,447.01",64.47,35.53
9,Optimal Product Depth,Weak Retention,1226,108,1118,47.37,52.93,2.00,"29,968.91",8.81,91.19
6,Optimal Product Depth,Moderate Retention,1190,89,1101,64.90,69.60,2.00,"68,562.09",7.48,92.52
7,Optimal Product Depth,Strong Retention,1208,24,1184,82.75,84.01,2.00,"33,964.02",1.99,98.01
8,Optimal Product Depth,Very Strong Retention,769,0,769,95.48,95.40,2.00,"90,271.66",0.00,100.00


## Final Customer Retention Profile

This step creates a final customer-level retention profile by combining risk segment, retention priority, relationship strength, and final retention strength tier.

This table will support dashboard-level customer prioritization and drill-down analysis.

In [43]:
# Step 7P: Create final customer retention profile

final_customer_retention_profile = df[
    [
        "CustomerId",
        "Geography",
        "Gender",
        "Age",
        "Tenure",
        "Balance",
        "EstimatedSalary",
        "NumOfProducts",
        "Credit_Card_Status",
        "Engagement_Status",
        "Product_Depth_Segment",
        "Customer_Risk_Segment",
        "Retention_Priority_Segment",
        "Relationship_Strength_Index",
        "Relationship_Strength_Tier",
        "Retention_Strength_Score",
        "Retention_Strength_Tier",
        "Exited"
    ]
].copy()

final_customer_retention_profile["Churn_Status"] = final_customer_retention_profile["Exited"].map({
    0: "Retained",
    1: "Churned"
})

final_customer_retention_profile = final_customer_retention_profile.drop(columns=["Exited"])

final_customer_retention_profile = final_customer_retention_profile.sort_values(
    by=["Retention_Strength_Score", "Relationship_Strength_Index", "Balance"],
    ascending=[True, True, False]
)

final_customer_retention_profile.head(10)

,CustomerId,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,NumOfProducts,Credit_Card_Status,Engagement_Status,Product_Depth_Segment,Customer_Risk_Segment,Retention_Priority_Segment,Relationship_Strength_Index,Relationship_Strength_Tier,Retention_Strength_Score,Retention_Strength_Tier,Churn_Status
729,15612525,France,Female,57,1,0.00,"131,372.38",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,15,Very Weak Relationship,0.00,Critical Retention Risk,Churned
3260,15768072,Spain,Female,33,2,0.00,"27,557.18",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,15,Very Weak Relationship,0.00,Critical Retention Risk,Churned
4220,15711314,Spain,Female,45,1,0.00,"125,939.22",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,15,Very Weak Relationship,0.00,Critical Retention Risk,Churned
4799,15656683,France,Male,52,1,0.00,"63,584.55",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,15,Very Weak Relationship,0.00,Critical Retention Risk,Churned
9044,15653347,Spain,Male,47,1,0.00,"128,882.66",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,15,Very Weak Relationship,0.00,Critical Retention Risk,Churned
9376,15657306,France,Female,47,2,0.00,"110,900.43",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,15,Very Weak Relationship,0.00,Critical Retention Risk,Churned
9635,15658768,France,Female,49,2,0.00,"65,466.93",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,15,Very Weak Relationship,0.00,Critical Retention Risk,Churned
2312,15724223,France,Female,55,5,0.00,"10,034.77",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,20,Very Weak Relationship,0.00,Critical Retention Risk,Churned
3975,15720071,France,Female,49,3,0.00,"61,820.41",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,20,Very Weak Relationship,0.00,Critical Retention Risk,Churned
6594,15808544,France,Female,40,3,0.00,"57,817.84",1,No Credit Card,Inactive Member,Single Product,Inactive Single-Product Risk,Already Churned,20,Very Weak Relationship,0.00,Critical Retention Risk,Churned


## Final Risk and Retention Strength Cross-Analysis

This step compares customer risk segments with final retention strength tiers.

This helps identify which risk groups have the weakest or strongest retention profiles.

In [44]:
# Step 7Q: Risk segment vs retention strength tier

risk_retention_strength_matrix = (
    df
    .groupby(["Customer_Risk_Segment", "Retention_Strength_Tier"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Retention_Strength_Score=("Retention_Strength_Score", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

risk_retention_strength_matrix["Churn_Rate"] = (
    risk_retention_strength_matrix["Churned_Customers"] /
    risk_retention_strength_matrix["Customer_Count"] * 100
).round(2)

risk_retention_strength_matrix["Retention_Rate"] = (
    risk_retention_strength_matrix["Retained_Customers"] /
    risk_retention_strength_matrix["Customer_Count"] * 100
).round(2)

for col in [
    "Avg_Retention_Strength_Score",
    "Avg_Relationship_Strength",
    "Avg_Balance",
    "Avg_Num_Products"
]:
    risk_retention_strength_matrix[col] = risk_retention_strength_matrix[col].round(2)

risk_retention_strength_matrix = risk_retention_strength_matrix.sort_values(
    by=["Customer_Risk_Segment", "Churn_Rate"],
    ascending=[True, False]
)

risk_retention_strength_matrix

,Customer_Risk_Segment,Retention_Strength_Tier,Customer_Count,Churned_Customers,Retained_Customers,Avg_Retention_Strength_Score,Avg_Relationship_Strength,Avg_Balance,Avg_Num_Products,Churn_Rate,Retention_Rate
0,At-Risk Premium,Critical Retention Risk,583,505,78,22.89,43.74,"114,207.54",1.14,86.62,13.38
4,At-Risk Premium,Weak Retention,1113,229,884,46.79,51.22,"115,175.29",1.32,20.58,79.42
1,At-Risk Premium,Moderate Retention,1069,158,911,64.94,66.76,"100,839.88",1.60,14.78,85.22
2,At-Risk Premium,Strong Retention,682,20,662,81.92,80.18,"131,395.09",1.07,2.93,97.07
3,At-Risk Premium,Very Strong Retention,5,0,5,98.00,70.00,"117,651.63",3.00,0.00,100.00
5,High Product Load Risk,Critical Retention Risk,75,75,0,24.17,31.27,"58,967.56",3.20,100.00,0.00
6,High Product Load Risk,Moderate Retention,49,39,10,63.49,64.59,"60,504.39",3.18,79.59,20.41
8,High Product Load Risk,Weak Retention,43,31,12,44.53,46.51,"55,797.16",3.14,72.09,27.91
7,High Product Load Risk,Strong Retention,15,0,15,83.33,58.33,"3,300.59",3.00,0.00,100.00
9,Inactive Single-Product Risk,Critical Retention Risk,882,479,403,17.43,36.89,"62,352.92",1.00,54.31,45.69


## Final Retention Scoring KPI Summary

This step prepares executive-level KPIs for the Retention Strength Scoring dashboard module.

These KPIs summarize customer retention strength, weak retention exposure, sticky customer share, and critical risk volume.

In [45]:
# Step 7R: Final retention scoring KPI summary

very_strong_retention_count = len(df[df["Retention_Strength_Tier"] == "Very Strong Retention"])
strong_retention_count = len(df[df["Retention_Strength_Tier"] == "Strong Retention"])
weak_retention_count = len(df[df["Retention_Strength_Tier"] == "Weak Retention"])
critical_retention_count = len(df[df["Retention_Strength_Tier"] == "Critical Retention Risk"])

strong_plus_retention_count = very_strong_retention_count + strong_retention_count
weak_plus_critical_count = weak_retention_count + critical_retention_count

avg_retention_strength_score = df["Retention_Strength_Score"].mean()
avg_relationship_strength = df["Relationship_Strength_Index"].mean()

retention_scoring_kpi_summary = pd.DataFrame({
    "KPI": [
        "Average Retention Strength Score",
        "Average Relationship Strength Index",
        "Strong Retention Customers",
        "Weak or Critical Retention Customers",
        "Sticky Customers",
        "Critical Retention Risk Customers"
    ],
    "Value": [
        f"{avg_retention_strength_score:.2f}",
        f"{avg_relationship_strength:.2f}",
        f"{strong_plus_retention_count:,}",
        f"{weak_plus_critical_count:,}",
        f"{sticky_customer_summary['Sticky Customers']:,}",
        f"{critical_retention_count:,}"
    ],
    "Business Meaning": [
        "Average customer-level retention strength score",
        "Average customer relationship strength",
        "Customers with strong or very strong retention profile",
        "Customers requiring retention monitoring or action",
        "Customers showing strong behavioral loyalty signals",
        "Customers with the weakest retention strength profile"
    ]
})

retention_scoring_kpi_summary

,KPI,Value,Business Meaning
0,Average Retention Strength Score,57.63,Average customer-level retention strength score
1,Average Relationship Strength Index,64.44,Average customer relationship strength
2,Strong Retention Customers,"2,727",Customers with strong or very strong retention...
3,Weak or Critical Retention Customers,"4,633",Customers requiring retention monitoring or ac...
4,Sticky Customers,"2,330",Customers showing strong behavioral loyalty si...
5,Critical Retention Risk Customers,"1,776",Customers with the weakest retention strength ...


## Final Retention Scoring Insight Table

This step creates a dashboard-ready insight table that connects analytical findings with dashboard usage.

The table helps translate retention scoring outputs into business actions.

In [46]:
# Step 7S: Final retention scoring insight table

highest_churn_tier = retention_strength_tier_churn.sort_values(
    by="Churn_Rate",
    ascending=False
).iloc[0]

strong_retention_tier = retention_strength_tier_churn.sort_values(
    by="Retention_Rate",
    ascending=False
).iloc[0]

retention_scoring_insight_table = pd.DataFrame({
    "Insight Area": [
        "Retention Strength Scoring",
        "Weak Retention Exposure",
        "Strong Retention Profile",
        "Risk and Strength Alignment",
        "Sticky Customer Identification",
        "Dashboard Priority"
    ],
    "Key Finding": [
        "Retention Strength Score combines engagement, product depth, relationship strength, card ownership, value, and churn status.",
        "Customers in weaker retention tiers show elevated churn risk and require closer monitoring.",
        "Strong retention tiers show better relationship strength and lower churn behavior.",
        "Risk segments and retention strength tiers together provide better prioritization than single-variable analysis.",
        "Sticky customers represent customers with strong behavioral loyalty signals.",
        "Retention scoring should be used as a core dashboard layer for segmentation, filtering, and prioritization."
    ],
    "Metric Evidence": [
        f"Average retention strength score: {avg_retention_strength_score:.2f}",
        f"Highest churn tier: {highest_churn_tier['Retention_Strength_Tier']} with {highest_churn_tier['Churn_Rate']}% churn",
        f"Best retention tier: {strong_retention_tier['Retention_Strength_Tier']} with {strong_retention_tier['Retention_Rate']}% retention",
        "Customer risk segment + retention strength tier matrix prepared",
        f"Sticky customers identified: {sticky_customer_summary['Sticky Customers']:,}",
        "Final customer retention profile prepared"
    ],
    "Dashboard Use": [
        "Retention strength KPI cards",
        "Weak retention warning panel",
        "Strong retention profile card",
        "Risk vs strength matrix",
        "Sticky customer profile section",
        "Customer-level retention scoring table"
    ]
})

retention_scoring_insight_table

,Insight Area,Key Finding,Metric Evidence,Dashboard Use
0,Retention Strength Scoring,"Retention Strength Score combines engagement, ...",Average retention strength score: 57.63,Retention strength KPI cards
1,Weak Retention Exposure,Customers in weaker retention tiers show eleva...,Highest churn tier: Critical Retention Risk wi...,Weak retention warning panel
2,Strong Retention Profile,Strong retention tiers show better relationshi...,Best retention tier: Very Strong Retention wit...,Strong retention profile card
3,Risk and Strength Alignment,Risk segments and retention strength tiers tog...,Customer risk segment + retention strength tie...,Risk vs strength matrix
4,Sticky Customer Identification,Sticky customers represent customers with stro...,"Sticky customers identified: 2,330",Sticky customer profile section
5,Dashboard Priority,Retention scoring should be used as a core das...,Final customer retention profile prepared,Customer-level retention scoring table


## Export Retention Scoring Outputs

This step exports the retention scoring and final segmentation outputs for dashboard development.

These files will support the Streamlit dashboard module focused on Retention Strength Scoring and Final Customer Segmentation.

In [47]:
# Step 7T: Export retention scoring outputs

RETENTION_STRENGTH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

retention_score_summary.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "retention_score_summary.csv",
    index=False
)

retention_strength_tier_distribution.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "retention_strength_tier_distribution.csv",
    index=False
)

retention_strength_tier_churn.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "retention_strength_tier_churn.csv",
    index=False
)

engagement_retention_strength_matrix.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "engagement_retention_strength_matrix.csv",
    index=False
)

product_depth_retention_strength_matrix.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "product_depth_retention_strength_matrix.csv",
    index=False
)

final_customer_retention_profile.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "final_customer_retention_profile.csv",
    index=False
)

risk_retention_strength_matrix.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "risk_retention_strength_matrix.csv",
    index=False
)

retention_scoring_kpi_summary.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "retention_scoring_kpi_summary.csv",
    index=False
)

retention_scoring_insight_table.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "retention_scoring_insight_table.csv",
    index=False
)

df.to_csv(
    RETENTION_STRENGTH_OUTPUT_DIR / "retention_strength_master_dataset.csv",
    index=False
)

print("Retention strength scoring outputs exported successfully.")
print(f"Output Directory: {RETENTION_STRENGTH_OUTPUT_DIR}")
print("Files Exported: 10")

Retention strength scoring outputs exported successfully.
Output Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\retention_strength_scoring
Files Exported: 10


## Retention Strength Analysis Insight Summary

This section summarizes the key retention strength findings generated in Notebook 07.

The analysis combines engagement behavior, product usage, credit card ownership, relationship strength, high-value status, and churn status to create a customer-level Retention Strength Score.

These findings help identify customers with strong loyalty signals, weak retention profiles, sticky behavior, and dashboard-ready retention monitoring segments.

In [48]:
# Step 7U: Retention strength analysis insight summary

retention_strength_analysis_summary = {
    "Analysis Area": "Retention Strength Scoring and Final Segmentation",
    "Input Dataset Rows": f"{df.shape[0]:,}",
    "Input Dataset Columns": df.shape[1],
    "Average Retention Strength Score": round(df["Retention_Strength_Score"].mean(), 2),
    "Median Retention Strength Score": round(df["Retention_Strength_Score"].median(), 2),
    "Strong or Very Strong Retention Customers": int(
        df[df["Retention_Strength_Tier"].isin(["Strong Retention", "Very Strong Retention"])].shape[0]
    ),
    "Weak or Critical Retention Customers": int(
        df[df["Retention_Strength_Tier"].isin(["Weak Retention", "Critical Retention Risk"])].shape[0]
    ),
    "Sticky Customers Identified": int(len(sticky_customers)),
    "Dashboard Module Ready": "Retention Strength Scoring and Final Segmentation"
}

retention_strength_analysis_summary

{'Analysis Area': 'Retention Strength Scoring and Final Segmentation',
 'Input Dataset Rows': '10,000',
 'Input Dataset Columns': 38,
 'Average Retention Strength Score': np.float64(57.63),
 'Median Retention Strength Score': 58.0,
 'Strong or Very Strong Retention Customers': 2727,
 'Weak or Critical Retention Customers': 4633,
 'Sticky Customers Identified': 2330,
 'Dashboard Module Ready': 'Retention Strength Scoring and Final Segmentation'}

## Dashboard Mapping for Retention Strength Scoring

This section maps Notebook 07 outputs to the Streamlit dashboard modules.

The exported files will support KPI cards, warning panels, scoring tables, retention tier visuals, sticky customer profiling, and final customer-level retention segmentation.

In [49]:
# Step 7V: Create dashboard mapping table for Notebook 07

retention_strength_dashboard_mapping = pd.DataFrame({
    "Dashboard Module": [
        "Retention Strength KPI Cards",
        "Retention Strength Tier Distribution",
        "Retention Tier Churn Analysis",
        "Engagement vs Retention Strength Matrix",
        "Product Depth vs Retention Strength Matrix",
        "Customer Risk vs Retention Strength Matrix",
        "Sticky Customer Profile Section",
        "Customer-Level Retention Scoring Table",
        "Retention Strength Insight Panel",
        "Final Master Dataset"
    ],
    "Notebook Output Used": [
        "retention_scoring_kpi_summary.csv",
        "retention_strength_tier_distribution.csv",
        "retention_strength_tier_churn.csv",
        "engagement_retention_strength_matrix.csv",
        "product_depth_retention_strength_matrix.csv",
        "customer_risk_retention_strength_matrix.csv",
        "sticky_customer_profile.csv",
        "final_customer_retention_profile.csv",
        "retention_scoring_insight_table.csv",
        "retention_strength_master_dataset.csv"
    ],
    "Dashboard Purpose": [
        "Display executive-level retention scoring KPIs",
        "Show customer distribution across retention strength tiers",
        "Compare churn and retention behavior by retention tier",
        "Analyze how engagement status connects with retention strength",
        "Analyze how product depth connects with retention strength",
        "Compare customer risk segments against retention strength tiers",
        "Highlight customers showing strong behavioral loyalty signals",
        "Provide customer-level scoring and prioritization table",
        "Summarize business insights and metric evidence",
        "Serve as the final dashboard-ready analytical dataset"
    ]
})

retention_strength_dashboard_mapping

,Dashboard Module,Notebook Output Used,Dashboard Purpose
0,Retention Strength KPI Cards,retention_scoring_kpi_summary.csv,Display executive-level retention scoring KPIs
1,Retention Strength Tier Distribution,retention_strength_tier_distribution.csv,Show customer distribution across retention st...
2,Retention Tier Churn Analysis,retention_strength_tier_churn.csv,Compare churn and retention behavior by retent...
3,Engagement vs Retention Strength Matrix,engagement_retention_strength_matrix.csv,Analyze how engagement status connects with re...
4,Product Depth vs Retention Strength Matrix,product_depth_retention_strength_matrix.csv,Analyze how product depth connects with retent...
5,Customer Risk vs Retention Strength Matrix,customer_risk_retention_strength_matrix.csv,Compare customer risk segments against retenti...
6,Sticky Customer Profile Section,sticky_customer_profile.csv,Highlight customers showing strong behavioral ...
7,Customer-Level Retention Scoring Table,final_customer_retention_profile.csv,Provide customer-level scoring and prioritizat...
8,Retention Strength Insight Panel,retention_scoring_insight_table.csv,Summarize business insights and metric evidence
9,Final Master Dataset,retention_strength_master_dataset.csv,Serve as the final dashboard-ready analytical ...


## Notebook 07 Final Conclusion

Notebook 07 completed the retention strength scoring and final segmentation analysis for the Customer Engagement & Product Utilization Analytics project.

The analysis created a dashboard-ready Retention Strength Score by combining customer engagement, credit card ownership, product depth, relationship strength, high-value status, and churn behavior.

The final results confirm that retention strength is not based on one factor alone. Strong retention behavior is driven by a combination of active engagement, stronger relationship strength, healthier product usage, and lower churn exposure.

Final analytical outcomes:

- Retention Strength Score was created at customer level.
- Retention Strength Tier segmentation was created.
- Weak and critical retention risk customers were identified.
- Strong and very strong retention customers were separated.
- Sticky customer profiles were identified.
- Engagement, product depth, and customer risk were compared against retention strength.
- Dashboard-ready KPI and insight tables were prepared.
- Final customer-level retention scoring dataset was exported successfully.

The exported outputs are now ready to support the Streamlit dashboard module:

`Retention Strength Scoring and Final Segmentation`

The next stage will focus on dashboard development, dashboard data integration, and Streamlit UI/UX implementation.

## Final Checkpoint Confirmation

This checkpoint confirms that Notebook 07 has been completed successfully and all required dashboard-ready outputs have been created.

In [50]:
# Step 7X: Final checkpoint confirmation for Notebook 07

print("Notebook 07 completed successfully.")
print("Analysis Area: Retention Strength Scoring and Final Segmentation")
print(f"Input Dataset Rows: {df.shape[0]:,}")
print(f"Input Dataset Columns: {df.shape[1]}")
print("Output Files Exported: 10")
print("Dashboard Module Ready: Retention Strength Scoring and Final Segmentation")
print("Final Dataset Exported: retention_strength_master_dataset.csv")

Notebook 07 completed successfully.
Analysis Area: Retention Strength Scoring and Final Segmentation
Input Dataset Rows: 10,000
Input Dataset Columns: 38
Output Files Exported: 10
Dashboard Module Ready: Retention Strength Scoring and Final Segmentation
Final Dataset Exported: retention_strength_master_dataset.csv
